# Demo — Fase 1: Novos Fetchers + DataLake

Notebook de validação visual da Fase 1 do `carteira_auto`.

**Pré-requisitos:**
```bash
pip install -e '.[dev]'
pip install jupyter matplotlib
```

**API Keys (opcionais — apenas para seções FRED e DDM):**
- `FRED_API_KEY`: gratuita em https://fred.stlouisfed.org/docs/api/api_key.html
- `DDM_API_KEY`: se disponível

Configure no `.env` na raiz do projeto ou via `os.environ` abaixo.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Descomenta e preenche se não tiver no .env
# os.environ['FRED_API_KEY'] = 'sua_chave_aqui'

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('ggplot')

print('✓ Ambiente configurado')

---
## 1. TesouroDiretoFetcher
Dados abertos — **sem API key necessária**

In [ ]:
from carteira_auto.data.fetchers import TesouroDiretoFetcher

tesouro = TesouroDiretoFetcher()
print('✓ TesouroDiretoFetcher inicializado')

In [ ]:
# Taxas atuais — snapshot do último dia disponível
df_atual = tesouro.get_current_rates()
print(f'Data de referência: {df_atual["data"].iloc[0].date()}')
df_atual[['tipo', 'vencimento', 'taxa_compra', 'taxa_venda', 'pu_compra']].sort_values('taxa_compra')

In [ ]:
# Títulos disponíveis
titulos = tesouro.get_available_titles()
print('Títulos disponíveis:', titulos)

In [ ]:
# Curva NTN-B (juros reais x prazo)
curva = tesouro.get_ntnb_curve()
print(f'Vértices na curva NTN-B: {len(curva)}')
print(curva[['vencimento', 'taxa_compra']].to_string(index=False))

In [ ]:
# Gráfico: Curva NTN-B
if not curva.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(curva['vencimento'], curva['taxa_compra'], 'o-', color='steelblue', linewidth=2, markersize=8)
    ax.set_title('Curva NTN-B — Taxa de Compra por Vencimento', fontsize=13)
    ax.set_xlabel('Vencimento')
    ax.set_ylabel('Taxa de Compra (% a.a.)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    for _, row in curva.iterrows():
        ax.annotate(f"{row['taxa_compra']:.2f}%",
                    xy=(row['vencimento'], row['taxa_compra']),
                    textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Histórico LFT — série completa
df_lft = tesouro.get_lft_history()
print(f'LFT: {len(df_lft)} registros históricos')
print(f'Período: {df_lft["data"].min().date()} → {df_lft["data"].max().date()}')
df_lft.tail(5)

In [ ]:
# Gráfico: LFT taxa de compra histórica
if not df_lft.empty:
    df_plot = df_lft.sort_values('data').tail(252 * 3)  # últimos 3 anos
    fig, ax = plt.subplots(figsize=(12, 5))
    for venc, g in df_plot.groupby('vencimento'):
        ax.plot(g['data'], g['taxa_compra'], label=str(pd.Timestamp(venc).year), alpha=0.8)
    ax.set_title('LFT — Taxa de Compra Histórica por Vencimento (últimos 3 anos)', fontsize=13)
    ax.set_xlabel('Data')
    ax.set_ylabel('Taxa (% a.a.)')
    ax.legend(title='Vencimento', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

---
## 2. FREDFetcher
Requer `FRED_API_KEY` configurada no `.env`

In [ ]:
from carteira_auto.data.fetchers import FREDFetcher

fred = FREDFetcher()

if not fred._api_key:
    print('⚠️  FRED_API_KEY não configurada — células FRED serão puladas')
    print('   Configure no .env e reinicie o kernel para ativar')
    FRED_OK = False
else:
    print(f'✓ FREDFetcher pronto')
    FRED_OK = True

# Séries disponíveis
series = FREDFetcher.list_series()
pd.DataFrame(series).T[['nome', 'unidade', 'frequencia']]

In [ ]:
if FRED_OK:
    # Bundle macro padrão
    bundle = fred.get_macro_bundle()
    print(f'Séries no bundle: {list(bundle.keys())}\n')
    for sid, df in bundle.items():
        ultimo = df.iloc[-1]
        print(f'  {sid:15s} | {len(df):5d} obs | último: {ultimo["value"]:8.4f} ({ultimo["date"].date()})')

In [ ]:
if FRED_OK:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()

    series_info = FREDFetcher.list_series()
    for i, (sid, df) in enumerate(bundle.items()):
        ax = axes[i]
        df_plot = df.set_index('date')['value'].tail(252 * 3)
        ax.plot(df_plot.index, df_plot.values, linewidth=1.5)
        nome = series_info.get(sid, {}).get('nome', sid)
        unidade = series_info.get(sid, {}).get('unidade', '')
        ax.set_title(f'{nome} ({sid})', fontsize=10)
        ax.set_ylabel(unidade, fontsize=9)
        ax.tick_params(axis='x', rotation=30)

    plt.suptitle('FRED — Bundle Macro (últimos 3 anos)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
if FRED_OK:
    # Spread 10Y-2Y (sinal de recessão)
    spread = fred.get_yield_curve_spread()
    df_spread = spread.set_index('date')['value'].tail(252 * 5)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(df_spread.index, df_spread.values, 0,
                    where=df_spread.values < 0, alpha=0.4, color='red', label='Invertida (risco recessão)')
    ax.fill_between(df_spread.index, df_spread.values, 0,
                    where=df_spread.values >= 0, alpha=0.3, color='green', label='Normal')
    ax.plot(df_spread.index, df_spread.values, color='navy', linewidth=1)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title('Spread Treasuries 10Y − 2Y (T10Y2Y) — Sinal de Recessão', fontsize=13)
    ax.set_ylabel('% (pontos percentuais)')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 3. CVMFetcher
Dados abertos — **sem API key necessária**

In [ ]:
from carteira_auto.data.fetchers import CVMFetcher

cvm = CVMFetcher()
print('✓ CVMFetcher inicializado')

In [ ]:
# Cadastro de companhias abertas
registro = cvm.get_company_registry()
print(f'Companhias abertas na CVM: {len(registro)}')
registro.head(10)

In [ ]:
# Mapeamento ticker → CNPJ (usa DDM como primário, CVM como fallback)
tickers_teste = ['PETR4', 'VALE3', 'ITUB4', 'WEGE3', 'ABEV3']
print('Mapeamento Ticker → CNPJ:')
for ticker in tickers_teste:
    cnpj = cvm.get_cnpj_by_ticker(ticker)
    print(f'  {ticker}: {cnpj or "não encontrado"}')

In [ ]:
# DFP — Demonstração Financeira Padronizada (DRE)
# Nota: O download pode demorar alguns segundos (arquivo ZIP ~10MB)
cnpj_petr = cvm.get_cnpj_by_ticker('PETR4')

if cnpj_petr:
    print(f'Buscando DFP DRE de PETR4 (CNPJ: {cnpj_petr}) — ano 2023...')
    try:
        df_dre = cvm.get_dfp(cnpj_petr, 2023, 'DRE')
        print(f'DRE: {len(df_dre)} linhas')
        df_dre.head(10)
    except Exception as e:
        print(f'Erro: {e}')
else:
    print('CNPJ da Petrobras não encontrado — verifique DDM_API_KEY ou conectividade')

---
## 4. DataLake — Ingestão e Consulta
Persiste dados dos fetchers no SQLite local

In [ ]:
from carteira_auto.data.lake import DataLake
from carteira_auto.config import settings
import tempfile
from pathlib import Path

# Usa diretório temporário para não sujar o lake de produção
DEMO_LAKE_DIR = Path(tempfile.mkdtemp()) / 'demo_lake'
DEMO_LAKE_DIR.mkdir()

lake = DataLake(DEMO_LAKE_DIR)
print(f'✓ DataLake criado em: {DEMO_LAKE_DIR}')

In [ ]:
# Ingestão macro via IngestMacroNode
from carteira_auto.core.nodes.ingest_nodes import IngestMacroNode
from carteira_auto.core.engine import PipelineContext

ctx = PipelineContext()
ctx['data_lake'] = lake

print('Executando IngestMacroNode (BCB + IBGE + FRED se configurado)...')
ctx = IngestMacroNode().run(ctx)
print(f'✓ Macro ingerido: {ctx["ingest_macro_count"]} registros')

In [ ]:
# Ingestão Tesouro Direto
from carteira_auto.core.nodes.ingest_nodes import IngestTesouroDiretoNode

print('Executando IngestTesouroDiretoNode (LFT, NTN-B, LTN, NTN-F)...')
ctx = IngestTesouroDiretoNode().run(ctx)
print(f'✓ Tesouro ingerido: {ctx["ingest_tesouro_count"]} registros')

In [ ]:
# Listar indicadores disponíveis no MacroLake
macro_lake = lake.macro
indicadores = macro_lake.list_indicators()
print(f'Indicadores no MacroLake: {len(indicadores)}')
for ind in sorted(indicadores):
    print(f'  {ind}')

In [ ]:
# Consultar Selic do BCB
df_selic = macro_lake.get('selic')
if df_selic is not None and not df_selic.empty:
    print(f'Selic BCB: {len(df_selic)} obs | último: {df_selic["value"].iloc[-1]:.4f}% ({df_selic["date"].iloc[-1].date()})')
    df_selic.tail()

In [ ]:
# Gráficos: indicadores macro do lake
indicadores_plot = [i for i in indicadores if not i.startswith('tesouro_') and not i.startswith('fred_')]

if indicadores_plot:
    n = min(len(indicadores_plot), 6)
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()

    for i, ind in enumerate(indicadores_plot[:n]):
        df_ind = macro_lake.get(ind)
        if df_ind is not None and not df_ind.empty:
            ax = axes[i]
            df_plot = df_ind.set_index('date')['value'].tail(252 * 3)
            ax.plot(df_plot.index, df_plot.values, linewidth=1.5)
            ax.set_title(ind, fontsize=10)
            ax.tick_params(axis='x', rotation=30)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('MacroLake — Indicadores BCB/IBGE (últimos 3 anos)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Consultar LFT do Tesouro (série mais longa disponível)
tesouro_series = [i for i in indicadores if i.startswith('tesouro_lft')]
print(f'Séries LFT no lake: {len(tesouro_series)}')

if tesouro_series:
    # Pega a de vencimento mais distante
    serie_nome = sorted(tesouro_series)[-1]
    df_lft_lake = macro_lake.get(serie_nome)
    print(f'\n{serie_nome}: {len(df_lft_lake)} obs')
    df_lft_lake.tail()

In [ ]:
# Gráfico: Todas as séries LFT do lake sobrepostas
lft_series = [i for i in indicadores if i.startswith('tesouro_lft')]

if lft_series:
    fig, ax = plt.subplots(figsize=(13, 5))
    for serie in sorted(lft_series):
        df_s = macro_lake.get(serie)
        if df_s is not None and not df_s.empty:
            label = serie.replace('tesouro_lft_', 'Venc. ')
            df_s.set_index('date')['value'].plot(ax=ax, label=label, alpha=0.8)
    ax.set_title('LFT — Taxa de Compra por Vencimento (MacroLake)', fontsize=13)
    ax.set_ylabel('% a.a.')
    ax.legend(title='Vencimento', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

---
## 5. IngestPricesNode — Preços + Screening
Busca preços de carteira + ~78 ações B3 para screening

In [ ]:
from carteira_auto.core.nodes.ingest_nodes import IngestPricesNode

node = IngestPricesNode(mode='daily')

# Ver universo de screening
print(f'Tickers de screening BR: {len(IngestPricesNode.SCREENING_TICKERS_BR)}')
print(IngestPricesNode.SCREENING_TICKERS_BR[:20], '...')

# Total de tickers que serão processados (sem carteira)
ctx_vazio = PipelineContext()
tickers_total = node._collect_tickers(ctx_vazio)
print(f'\nTotal de tickers (sem carteira): {len(tickers_total)}')
print('Benchmarks:', [t for t in tickers_total if t.startswith('^')])
print('Crypto:', [t for t in tickers_total if '-USD' in t])
print('Commodities:', [t for t in tickers_total if '=F' in t])

In [ ]:
# Ingestão de preços — modo daily (últimos 5 dias úteis)
# Aviso: esta célula faz chamadas reais ao Yahoo Finance (~1-2 min)
print('Buscando preços (modo daily — últimos 5 dias)...')
print('Isso pode levar 1-2 minutos...\n')

ctx2 = PipelineContext()
ctx2['data_lake'] = lake
ctx2 = IngestPricesNode(mode='daily').run(ctx2)
print(f'\n✓ Preços ingeridos: {ctx2["ingest_prices_count"]} registros')

In [ ]:
# Consultar preços do lake
price_lake = lake.prices
tickers_no_lake = price_lake.list_tickers()
print(f'Tickers com preços no lake: {len(tickers_no_lake)}')
print(sorted(tickers_no_lake)[:20], '...' if len(tickers_no_lake) > 20 else '')

In [ ]:
# Comparar retornos dos benchmarks
benchmarks = ['^BVSP', '^GSPC', '^IXIC']
nomes = {'^BVSP': 'IBOV', '^GSPC': 'S&P 500', '^IXIC': 'Nasdaq'}

fig, ax = plt.subplots(figsize=(12, 5))
for ticker in benchmarks:
    df_b = price_lake.get(ticker)
    if df_b is not None and not df_b.empty and 'close' in df_b.columns:
        serie = df_b.set_index('date')['close'].sort_index().tail(252)
        (serie / serie.iloc[0] * 100).plot(ax=ax, label=nomes.get(ticker, ticker))

if ax.lines:
    ax.set_title('Benchmarks — Retorno Relativo (base 100)', fontsize=13)
    ax.set_ylabel('Índice (base 100)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Dados insuficientes para gráfico (rode no modo full para histórico completo)')

---
## 6. Backward Compatibility — Pipeline Existente
Garante que as pipelines da v0.1.0 continuam funcionando

In [ ]:
from carteira_auto.core.registry import PIPELINE_PRESETS, create_engine

print('Pipelines registradas:')
for name in PIPELINE_PRESETS:
    print(f'  • {name}')

In [ ]:
# Verifica que todos os IngestNodes estão disponíveis
from carteira_auto.core.nodes import (
    IngestPricesNode,
    IngestMacroNode,
    IngestFundamentalsNode,
    IngestNewsNode,
    IngestCVMNode,
    IngestTesouroDiretoNode,
)

nodes = [IngestPricesNode, IngestMacroNode, IngestFundamentalsNode,
         IngestNewsNode, IngestCVMNode, IngestTesouroDiretoNode]

print('IngestNodes disponíveis:')
for NodeClass in nodes:
    n = NodeClass()
    print(f'  ✓ {NodeClass.__name__} (name="{n.name}", deps={n.dependencies})')

In [ ]:
# Verifica todos os fetchers disponíveis
from carteira_auto.data.fetchers import (
    YahooFinanceFetcher,
    BCBFetcher,
    IBGEFetcher,
    DDMFetcher,
    CVMFetcher,
    FREDFetcher,
    TesouroDiretoFetcher,
)

fetchers = [YahooFinanceFetcher, BCBFetcher, IBGEFetcher,
            DDMFetcher, CVMFetcher, FREDFetcher, TesouroDiretoFetcher]

print('Fetchers disponíveis:')
for FetcherClass in fetchers:
    print(f'  ✓ {FetcherClass.__name__}')

---
## 7. Resumo do Estado do DataLake

In [ ]:
indicadores_final = macro_lake.list_indicators()
tickers_final = price_lake.list_tickers() if hasattr(price_lake, 'list_tickers') else []

print('=' * 50)
print('RESUMO DO DATALAKE (demo)')
print('=' * 50)
print(f'MacroLake — indicadores: {len(indicadores_final)}')
print(f'  BCB:             {len([i for i in indicadores_final if not i.startswith(("tesouro_","fred_","ddm_","ipca_ibge","pib"))])}')
print(f'  Tesouro Direto:  {len([i for i in indicadores_final if i.startswith("tesouro_")])}')
print(f'  FRED:            {len([i for i in indicadores_final if i.startswith("fred_")])}')
print(f'  DDM:             {len([i for i in indicadores_final if i.startswith("ddm_")])}')
print(f'PriceLake — tickers: {len(tickers_final)}')
print('=' * 50)
print('\n✅ Fase 1 validada com sucesso!')